# Developing an exceptions module

Created: 2026-03-21

Like most Python packages, `estatjp` uses many dependencies, all of which can raise exceptions. Even with tracebacks, locating the source of an exception can be frustrating and time consuming. The purpose of including an exceptions module is to bring exception handling in estatjp under one roof in order to properly deal with any exceptions and simplify users' debugging tasks.

Technical references consulted for the code below:

   * [Built-in Exceptions](https://docs.python.org/3.14/library/exceptions.html)
   * [8. Errors and Exceptions](https://docs.python.org/3.14/tutorial/errors.html#errors-and-exceptions)
   * [8.4. The try statement](https://docs.python.org/3.14/reference/compound_stmts.html#the-try-statement)
   * [4.3. Exceptions](https://docs.python.org/3.14/reference/executionmodel.html#exceptions)
   * [Python 3: Deep Dive (Part 4 - OOP): カスタム例外 (セクション12-3/15)](https://note.com/hafnium/n/nbb6179633a5e)

## Subtasks to focus development:

### API url checking

E-Stat API calls allow users to request responses in one of three formats: XML, JSON, and CSV. The choice is specified in the API request url. `estatpy.api.get_csv_data()` should check the url before sending its request to e-Stat and raise an exception if the url is not requesting CSV. Other errors returned by the server should also be handled in a local exception.

### Response stream format checking

In addition to the XML, JSON, and CSV stream formats via API calls, e-Stat also provides file downloads in the following formats. For an example, click on "Download" in this database of [machinery orders](https://www.e-stat.go.jp/en/dbview?sid=0003355268).

    * XLSX
    * CSV cross tabulation in shift-JIS encoding
    * CSV cross tabulation in utf-8 encoding and byte-order marking (BOM)
    * CSV cross tabulation in utf-8 without the BOM
    * CSV column-oriented format in shift-JIS

Mismatches stream or file format with function code should be caught before processing the stream or file.

### Tests

This package doesn't have any tests yet. Develop a test for the presence of the `appId=` string in the API url's query string.

## Defining a first exception

Even with traceback, finding the source of an exception can be exasperating, particularly when a raised exception does not state its source. Plus, <a href="https://docs.python.org/3.14/reference/executionmodel.html#exceptions">the Python Language Reference</a> contains the following note:

***

Exception messages are not part of the Python API. Their contents may change from one version of Python to the next without warning and should not be relied on by code which will run under multiple versions of the interpreter.

***

To provide package users with clearer exception information, exception handlers that require communication to the user will raise a local exception that will add local information and then raise its base class exception.

### Define AppIDError class

In [1]:
# The case where the query string of a url to be used for an api request is missing the `appId=` string.
from datetime import datetime

class estatjpError(Exception):
    """Exception for when API request url is lacking a required 'appId=' string. Ref: <https://note.com/hafnium/n/nbb6179633a5e>"""
    internal_err_msg = "estatjp error."

    def __init__(self, *args, user_err_msg=None):
        # use first argument as the internal error message if provided
        if args:
            self.internal_err_msg = args[0]
            super().__init__(*args)
        else:
            super().__init__(self.internal_err_msg)

        if user_err_msg is not None:
            self.user_err_msg = user_err_msg
    
    def log_exception(self):
        exception_data = {
            "type": type(self).__name__,
            "message": self.internal_err_msg,
            "args": self.args[1:],
            "timestamp": datetime.now(datetime.timezone.utc).isoformat()
        }
        print(f"LOG_EXCEPTION: {exception_data}")
    
    def get_notes(self):
        li = self.__dict__.get("__notes__")
        return ' '.join(li)

class AppIDError(estatjpError):
    user_err_msg = "The API request url is lacking a required 'appId=' string."
    pass

def CheckUrl(url):
    url_split = url.split("appId=")
    if len(url_split) != 2:
        e = AppIDError()
        e.add_note("CheckUrl: missing string 'appId=',\nurl=" + url)
        raise e
    return True
    
testurl1 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'
testurl2 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'

def testfunction(url):
    try:
        return CheckUrl(url)
    except AppIDError as ex:
        print("testfunction error:\n", ex.get_notes())
    return "OK"


print(testfunction(testurl2))

print(testfunction(testurl1))


True
testfunction error:
 CheckUrl: missing string 'appId=',
url=http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0
OK


### Test estatjp.exceptions.AppIDError

In [ ]:
testurl1 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'
testurl2 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'

from estatjp import exceptions as xs

def CheckUrl2(url):
    url_split = url.split("appId=")
    if len(url_split) != 2:
        err_msg = "From CheckUrl: " + url
        try:
            raise xs.AppIDError()
        except xs.AppIDError as e:
            e.add_note(err_msg)
            raise
    return True


def testfunction2(url):
    result = False
    try:
        result = CheckUrl2(url)
    except xs.AppIDError as ex:
        print("testfunction2 AppIDError error: ", ex)
        result = ex
    except Exception as xc:
        print("testfunction2 Exception error: ", xc)
        result = False
    return result


print(testfunction2(testurl2))

print(testfunction2(testurl1))


True
testfunction2 AppIDError error:  estatjp error.
False


## Script for automated testing

This first script checks the logic for the test. The erroneous url should raise an exceptions.AppIDError.

In [1]:
urltest1 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'
urltest2 = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'

from estatjp import exceptions as xs
from estatjp import api

def CheckForAppIdString(url):
    url_split = url.split("appId=")
    if len(url_split) != 2:
        err_msg = "From CheckForAppIdString: " + url
        try:
            raise xs.AppIDError()
        except xs.AppIDError as e:
            e.add_note(err_msg)
            raise
    return True

def driver_function(url):
    result = False
    try:
        result = api.get_csv_data(url)
    except xs.AppIDError as ex:
        result = ex
    except Exception as xc:
        result = xc
    return result

def test_AppIDError_true():
    res = driver_function(urltest1)
    assert isinstance(res,xs.AppIDError)

test_AppIDError_true()
